# COVID-19 Trends Dashboard — Interactive Analysis

An exploratory walkthrough of global COVID-19 patterns using the Our World in Data dataset.

**Questions explored:**
1. How did case rates evolve globally? Can we identify distinct waves?
2. Which countries were hit hardest, per-capita?
3. How did vaccination coverage vary by wealth and geography?
4. Does the reported COVID death count tell the full story? (Spoiler: no.)
5. How did government response (stringency) track with case loads?
6. Which variant drove which wave?
7. Can simple models — compartmental (SIR/SEIR) and statistical (Prophet) — forecast a wave?

---

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the src/ and models/ packages importable from the notebook
sys.path.insert(0, str(Path('..').resolve() / 'src'))
sys.path.insert(0, str(Path('..').resolve() / 'models'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_data, split_countries_and_aggregates
from analysis import add_case_fatality_rate, rolling_average, top_n_by_metric, summarize_country
from visualizations import apply_style

apply_style()
pd.set_option('display.max_columns', 80)

## 1. Load and explore the data

The dataset mixes country rows with aggregate rows (continents, income groups, "World"). Aggregate rows have no `continent` value — we split them out so we never accidentally double-count.

In [ ]:
df = load_data()
countries, aggregates = split_countries_and_aggregates(df)

print(f"Total rows:   {len(df):,}")
print(f"Countries:    {countries['location'].nunique()}")
print(f"Aggregates:   {sorted(aggregates['location'].unique())}")
print(f"Date range:   {df['date'].min().date()} → {df['date'].max().date()}")

## 2. Global headline numbers

In [ ]:
world = aggregates[aggregates['location'] == 'World'].sort_values('date')

def last_valid(col):
    v = world[col].dropna()
    return v.iloc[-1] if not v.empty else None

summary = pd.Series({
    'Total cases': f"{last_valid('total_cases'):,.0f}",
    'Total deaths': f"{last_valid('total_deaths'):,.0f}",
    'Cases per million': f"{last_valid('total_cases_per_million'):,.0f}",
    'Deaths per million': f"{last_valid('total_deaths_per_million'):,.0f}",
    '% fully vaccinated': f"{last_valid('people_fully_vaccinated_per_hundred'):.1f}%",
})
summary.to_frame('value')

## 3. Identifying global waves

The smoothed global case curve shows several distinct waves. The biggest by far is the Omicron wave in early 2022, which peaked at over 3 million cases per day globally. Note the spike in late 2022 — that's largely China's reporting jump after dropping zero-COVID policies.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
ts = world.set_index('date')['new_cases_smoothed']
ax.fill_between(ts.index, ts.values, alpha=0.25)
ax.plot(ts.index, ts.values, linewidth=1.8)
ax.set_title('Global daily new cases (7-day rolling avg)')
ax.set_ylabel('Cases per day')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.show()

## 4. Per-capita rankings: who got hit hardest?

Looking at raw case counts is misleading because of population differences. Per-million metrics reveal that small, well-tested European countries (and South Korea) actually report the highest per-capita case rates. This says more about testing infrastructure than infection prevalence.

In [ ]:
top = top_n_by_metric(countries, 'total_cases_per_million', n=15, min_population=1_000_000)
rankings = (countries[countries['location'].isin(top)]
            .dropna(subset=['total_cases_per_million'])
            .groupby('location', observed=True)
            .tail(1)
            .sort_values('total_cases_per_million', ascending=False)
            [['location', 'total_cases_per_million', 'total_deaths_per_million']]
            .reset_index(drop=True))
rankings

## 5. Vaccination equity — wealth vs coverage

Plotting GDP per capita against vaccination rate reveals a striking divide. High-income countries clustered around 70–90% coverage, while many low-income countries (especially in sub-Saharan Africa) never crossed 30%.

In [ ]:
valid = countries.dropna(subset=['people_fully_vaccinated_per_hundred', 'gdp_per_capita', 'continent'])
idx = valid.groupby('location', observed=True)['date'].idxmax()
latest = valid.loc[idx]

# Group by income brackets
latest['income_bracket'] = pd.cut(
    latest['gdp_per_capita'],
    bins=[0, 5000, 15000, 35000, 1e6],
    labels=['Low (<$5k)', 'Lower-mid ($5–15k)', 'Upper-mid ($15–35k)', 'High (>$35k)']
)

summary = (latest.groupby('income_bracket', observed=True)['people_fully_vaccinated_per_hundred']
                .agg(['mean', 'median', 'count'])
                .round(1))
summary.columns = ['Mean %', 'Median %', 'Countries']
summary

## 6. The mortality undercount — excess mortality vs reported deaths

This is one of the most important comparisons in the dataset. Excess mortality (deaths above the historical baseline) captures the pandemic's *true* toll, including uncounted COVID deaths and indirect deaths from overwhelmed health systems. For many countries — Russia, Bulgaria, Serbia, South Africa — excess mortality is **2–3× higher** than the reported COVID death count, suggesting massive undercounting.

In [ ]:
def latest_valid_value(group, col):
    v = group[col].dropna()
    return v.iloc[-1] if not v.empty else np.nan

grouped = countries.groupby('location', observed=True)
comp = pd.DataFrame({
    'excess_mortality_per_M': grouped.apply(lambda g: latest_valid_value(g, 'excess_mortality_cumulative_per_million')),
    'reported_deaths_per_M': grouped.apply(lambda g: latest_valid_value(g, 'total_deaths_per_million')),
    'population': grouped['population'].max(),
}).reset_index().dropna(subset=['excess_mortality_per_M', 'reported_deaths_per_M'])

comp = comp.query('population >= 3_000_000').copy()
comp['ratio'] = comp['excess_mortality_per_M'] / comp['reported_deaths_per_M']
comp.sort_values('ratio', ascending=False).head(15)[['location', 'excess_mortality_per_M', 'reported_deaths_per_M', 'ratio']]

## 7. Country deep-dive

Pick any country to get a quick summary.

In [ ]:
for country in ['United States', 'Brazil', 'India', 'Japan', 'South Africa']:
    s = summarize_country(countries, country)
    print(f"\n{s['country']}")
    print(f"  Population:      {s['population']:>15,.0f}")
    print(f"  Total cases:     {s['total_cases']:>15,.0f}")
    print(f"  Total deaths:    {s['total_deaths']:>15,.0f}")
    print(f"  Deaths/M:        {s['deaths_per_million']:>15,.0f}")
    print(f"  Fully vax:       {s['pct_fully_vaccinated']:>14.1f}%")
    print(f"  Peak day:        {s['peak_daily_cases_date']} ({s['peak_daily_cases']:,.0f} cases)")

---
# Extended analysis

The sections below cover the features added on top of the original dashboard: the variant timeline, hospitalization data, wave forecasting (SIR/SEIR plus Prophet), and the choropleth maps.

## 8. Which variant drove which wave?

`src/variants.py` carries approximate windows for when each variant dominated the global picture, plus WHO designation dates. The `overlay_variants` helper shades these onto any matplotlib date axis. Note these windows are for *annotation* — actual variant timing differed substantially by country.

In [ ]:
from variants import overlay_variants, variant_table

variant_table()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
ts = world.set_index('date')['new_cases_smoothed']
ax.fill_between(ts.index, ts.values, alpha=0.20)
ax.plot(ts.index, ts.values, linewidth=1.8)

# Overlay the variant bands — call AFTER plotting so x-limits are set
overlay_variants(ax, alpha=0.13, text_y=0.9)

ax.set_title('Global daily new cases — with variant-dominance bands')
ax.set_ylabel('Cases per day')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.show()

## 9. Hospitalization & ICU occupancy

OWID's hospitalization columns are sparsely populated — only ~30–40 mostly-high-income countries ever reported them. Here's how to check coverage and plot what's available.

In [ ]:
# Which countries actually reported hospitalization data?
hosp_coverage = (countries.groupby('location', observed=True)['hosp_patients_per_million']
                          .apply(lambda s: s.notna().sum()))
hosp_coverage = hosp_coverage[hosp_coverage > 0].sort_values(ascending=False)
print(f"{len(hosp_coverage)} countries reported hospital occupancy data.")
hosp_coverage.head(15).to_frame('days_reported')

In [ ]:
selected = ['United States', 'United Kingdom', 'France', 'Italy']
fig, ax = plt.subplots(figsize=(13, 5.5))
for name in selected:
    cs = countries[countries['location'] == name].sort_values('date')
    ax.plot(cs['date'], cs['icu_patients_per_million'], label=name, linewidth=1.6)
overlay_variants(ax, text_y=0.93)
ax.set_title('ICU patients per million — variant bands shaded')
ax.set_ylabel('ICU patients per million')
ax.legend()
plt.show()

## 10. Forecasting a wave: SIR / SEIR and Prophet

Two model families, fit to a single wave window:

- **`models/epidemic_models.py`** — SIR and SEIR compartmental models (mechanistic; fitted β, γ, R₀ are interpretable).
- **`models/prophet_model.py`** — Prophet, a statistical additive model (trend + weekly seasonality). Optional dependency: if `prophet` isn't installed, this section's Prophet cell just prints a notice and the rest of the notebook runs fine.

All three expose the same `.forecast()` contract. Below we fit them to the US Delta wave and project forward, then compare to what actually happened.

The key lesson: the mechanistic models **overshoot** (constant parameters can't capture behaviour change), while Prophet tends to **over-trust the local trend**. Seeing them miss differently is the point.

In [ ]:
from epidemic_models import fit_sir, fit_seir, slice_wave

# Extract the US Delta wave window
dates, values, pop = slice_wave(countries, 'United States', '2021-06-15', '2021-10-15')

sir = fit_sir(dates, values, population=pop)
seir = fit_seir(dates, values, population=pop)

print(sir.summary())
print()
print(seir.summary())

In [ ]:
sir_fc = sir.forecast(horizon_days=45)
seir_fc = seir.forecast(horizon_days=45)

# Actuals: fit window + 45 days, so we can eyeball the forecast
full = countries[countries['location'] == 'United States'].sort_values('date')
obs = full[(full['date'] >= '2021-06-15') & (full['date'] <= '2021-11-29')]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(obs['date'], obs['new_cases_smoothed'], color='#333', linewidth=2.2, label='Actual (7d avg)')
ax.plot(sir_fc['date'], sir_fc['new_cases'], label=f'SIR (R0={sir.r0:.2f})', linewidth=1.7)
ax.plot(seir_fc['date'], seir_fc['new_cases'], label=f'SEIR (R0={seir.r0:.2f})', linewidth=1.7)
ax.axvline(pd.Timestamp('2021-10-15'), color='grey', linestyle=':', linewidth=1.4)
ax.set_title('US Delta wave — SIR/SEIR fit + 45-day forecast vs actuals')
ax.set_ylabel('New cases per day')
ax.legend()
plt.show()

Now the same wave with **Prophet**. It's a statistical curve-fit — no R₀, no compartments — so it fails differently from SIR/SEIR. Prophet is an optional dependency; the cell below degrades gracefully if it isn't installed.

In [ ]:
from prophet_model import fit_prophet, HAVE_PROPHET

if not HAVE_PROPHET:
    print('prophet not installed — skipping. Install with: pip install prophet')
else:
    prophet_fit = fit_prophet(dates, values)
    prophet_fc = prophet_fit.forecast(horizon_days=45)
    print(prophet_fit.summary())

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(obs['date'], obs['new_cases_smoothed'], color='#333', linewidth=2.2,
            label='Actual (7d avg)')
    ax.plot(sir_fc['date'], sir_fc['new_cases'], label=f'SIR (R0={sir.r0:.2f})',
            linewidth=1.5, alpha=0.8)
    ax.plot(seir_fc['date'], seir_fc['new_cases'], label=f'SEIR (R0={seir.r0:.2f})',
            linewidth=1.5, alpha=0.8)
    ax.plot(prophet_fc['date'], prophet_fc['new_cases'], label='Prophet',
            linewidth=1.7, linestyle='--')
    fc_only = prophet_fc[prophet_fc['kind'] == 'forecast']
    ax.fill_between(fc_only['date'], fc_only['lower'], fc_only['upper'],
                    alpha=0.15, label='Prophet 90% interval')
    ax.axvline(pd.Timestamp('2021-10-15'), color='grey', linestyle=':', linewidth=1.4)
    ax.set_title('US Delta wave — SIR / SEIR / Prophet, 45-day forecast vs actuals')
    ax.set_ylabel('New cases per day')
    ax.legend()
    plt.show()

## 11. Choropleth world maps

`src/choropleth.py` builds Plotly choropleths using OWID's `iso_code` column — no GeoPandas needed. `build_choropleth` returns a Plotly figure you can display inline or drop into the Dash app.

In [ ]:
from choropleth import build_choropleth

fig = build_choropleth(countries, 'total_deaths_per_million')
fig.show()

In [ ]:
# Snapshot as of a specific date — uses each country's latest value on or before that date
fig = build_choropleth(countries, 'people_fully_vaccinated_per_hundred', as_of=pd.Timestamp('2021-07-01'))
fig.show()

## Next steps

Run the full dashboard script to generate the complete set of static charts:

```bash
python src/dashboard.py
```

Or launch the interactive Dash app for live country/date filtering, the choropleth map, and on-the-fly SIR/SEIR/Prophet forecasting:

```bash
python interactive/app.py
```